In [ ]:
import pandas as pd
import numpy as np

### test df.assign and df.pipe

In [ ]:
suspension_columns = ["Status", "Suspension reason name", "Description"]
AUTHOR_PREFIX = "Ri"

df_assign = pd.DataFrame(columns=suspension_columns, index=[0])

In [ ]:
def generate_name(
    prefix: str = "",
    index_series: pd.Index | pd.DataFrame | pd.Series = None,
    name: str = "",
    post_fix: str = ""
    ) -> pd.Series:

    if index_series is None:
        index_series = pd.Series([0])
    if not isinstance(index_series, pd.Index):
        index_series = pd.Series(index_series).index

    return prefix + name + (index_series + 1).astype(str).str.zfill(2) + post_fix

#### df.assign

In [ ]:
status_kwargs = {
    "Status": df_assign["Status"].map(lambda x: ["Active", "Inactive"]),
}
df_assign = df_assign.assign(**status_kwargs)
df_assign = df_assign.explode("Status", ignore_index=True)

kwargs2 = {
    "Suspension reason name": generate_name(AUTHOR_PREFIX, df_assign.index, "SuspensionReasonName"),
    "Description": generate_name(AUTHOR_PREFIX, df_assign.index, "SuspensionReasonDescription")
}
df_assign = df_assign.assign(**kwargs2)
df_assign

#### df.pipe

In [ ]:
df_pipe = pd.DataFrame(columns=suspension_columns, index=[0])
df_pipe = df_pipe.assign(**status_kwargs)
df_pipe = df_pipe.explode("Status", ignore_index=True)

df_pipe

In [ ]:
def generate_name_pipe_version(
    colum_name: str = "",
    prefix: str = "",
    df: pd.DataFrame = None,
    name: str = "",
    postfix: str = ""
    ) -> pd.DataFrame:

    df[colum_name] = prefix + name + (df.index + 1).astype(str).str.zfill(2) + postfix

    return df

In [ ]:
def generate_name_pipe_version_2(
    df: pd.DataFrame = None,
    colum_name: str = "",
    prefix: str = "",
    name: str = "",
    postfix: str = ""
    ) -> pd.DataFrame:

    df[colum_name] = prefix + name + (df.index + 1).astype(str).str.zfill(2) + postfix

    return df

In [ ]:
df_pipe = (
    df_pipe
    .pipe(
        (generate_name_pipe_version, "df"),
        colum_name="Suspension reason name",
        prefix=AUTHOR_PREFIX,
        name="SuspensionReasonName"
    )
    .pipe(
        (generate_name_pipe_version, "df"),
        colum_name="Description",
        prefix=AUTHOR_PREFIX,
        name="SuspensionReasonDescription"
    )
)
df_pipe

In [ ]:
df_pipe = (
    df_pipe
    .pipe(
        generate_name_pipe_version_2,
        colum_name="Suspension reason name",
        prefix=AUTHOR_PREFIX,
        name="SuspensionReasonName"
    )
    .pipe(
        generate_name_pipe_version_2,
        colum_name="Description",
        prefix=AUTHOR_PREFIX,
        name="SuspensionReasonDescription"
    )
)
df_pipe

In [ ]:
data = [[8000, 1000], [9500, np.nan], [5000, 2000]]
df = pd.DataFrame(data, columns=["Salary", "Others"])
df


In [ ]:
def subtract_federal_tax(df):
    return df * 0.9
def subtract_state_tax(df, rate):
    return df * (1 - rate)
def subtract_national_insurance(df, rate, rate_increase):
    new_rate = rate + rate_increase
    return df * (1 - new_rate)

In [ ]:
subtract_national_insurance(
    subtract_state_tax(subtract_federal_tax(df), rate=0.12),
    rate=0.05,
    rate_increase=0.02,
)

In [ ]:
(
    df.pipe(subtract_federal_tax)
    .pipe(subtract_state_tax, rate=0.12)
    .pipe((subtract_national_insurance, "df"), rate=0.05, rate_increase=0.02)
)

In [ ]:
a = "RiHonoBatch13" * 510
print(a[:518])